In [1]:
import json
import os
import numpy as np

In [2]:
from utils.pc_utils import random_sampling

In [3]:
scan_val = 'data/ScanRefer_filtered_val_with_id.json'

In [4]:
with open(scan_val, 'r') as f:
    val_data = json.load(f)

In [5]:
import pandas as pd
df = pd.read_csv('data/data_val.csv')

In [6]:
cls = df['class'].to_numpy()
nel = df['nel'].to_numpy()

In [7]:
import ast
nel_label = [ast.literal_eval(nel_label) for nel_label in nel]

In [8]:
cls

array([ 2,  2,  2, ...,  8, 17,  8])

In [9]:
nel_label

[[2, 4, 17],
 [2, 17],
 [2, 4],
 [0, 17],
 [17],
 [4, 8, 17],
 [4, 5, 8, 17],
 [4, 17],
 [9, 17],
 [2, 4, 8, 17],
 [6, 9],
 [4, 17],
 [6, 9, 17],
 [17],
 [2, 4],
 [2, 4],
 [2, 4],
 [9, 17],
 [0, 17],
 [2, 4],
 [2, 4],
 [0, 17],
 [17],
 [4, 17],
 [4, 17],
 [0, 9, 17],
 [6, 9, 17],
 [9, 17],
 [0, 17],
 [9, 17],
 [17],
 [2, 4, 17],
 [2],
 [2, 4, 17],
 [2, 4, 5],
 [2],
 [2, 4],
 [2, 4],
 [4, 17],
 [2, 4],
 [2, 4],
 [2],
 [0, 6, 17],
 [6, 9, 17],
 [0, 9, 17],
 [2, 4],
 [0, 2, 17],
 [2, 4],
 [17],
 [2],
 [2, 4],
 [2, 4],
 [2, 4],
 [0, 2, 17],
 [4, 17],
 [2, 4],
 [2, 4, 5, 17],
 [2, 4],
 [4, 17],
 [2, 4],
 [5, 17],
 [2, 4],
 [2, 4],
 [0, 6, 17],
 [0, 17],
 [2, 4, 17],
 [2, 4],
 [17],
 [6, 9, 15],
 [2, 4],
 [4, 17],
 [2, 4],
 [2, 4],
 [2, 4],
 [2, 4],
 [0, 2, 4, 9, 17],
 [10, 17],
 [2, 17],
 [10, 17],
 [2, 10],
 [10, 17],
 [2, 17],
 [2, 4],
 [2, 17],
 [2],
 [10, 17],
 [4, 17],
 [10, 17],
 [2, 4],
 [4, 17],
 [2, 4],
 [2, 4, 17],
 [4],
 [2, 4, 17],
 [2, 4],
 [2, 4],
 [4, 17],
 [2, 17],
 [2, 17],

In [10]:
DC = {5: 2, 23: 17, 8: 5, 40: 17, 9: 6, 7: 4, 39: 17, 18: 17, 11: 8, 29: 17, 3: 0, 14: 10, 15: 17, 27: 17, 6: 3, 34: 15, 35: 17, 4: 1, 10: 7, 19: 17, 16: 11, 30: 17, 33: 14, 37: 17, 21: 17, 32: 17, 25: 17, 17: 17, 24: 12, 28: 13, 36: 16, 12: 9, 38: 17, 20: 17, 26: 17, 31: 17, 13: 17}

In [11]:
num_target = []
num_nel = []
for i, data in enumerate(val_data[:1]):
    print(data)
    scene_id = data["scene_id"]
    object_id = int(data["object_id"])
    mesh_vertices = np.load(os.path.join('data/scannet/pointgroup_data', scene_id) + "_aligned_vert.npy")  # axis-aligned
    instance_labels = np.load(os.path.join('data/scannet/pointgroup_data', scene_id) + "_ins_label_pg.npy")
    semantic_labels = np.load(os.path.join('data/scannet/pointgroup_data', scene_id) + "_sem_label_pg.npy")
    instance_bboxes = np.load(os.path.join('data/scannet/pointgroup_data', scene_id) + "_aligned_bbox.npy")
    MEAN_COLOR_RGB = np.array([109.8, 97.2, 83.8])
    point_cloud = mesh_vertices[:, 0:6]
    point_cloud[:, 3:6] = (point_cloud[:, 3:6] - MEAN_COLOR_RGB) / 256.0
    pcl_color = point_cloud[:, 3:6]

    point_cloud, choices = random_sampling(point_cloud, 50000, return_choices=True)
    instance_labels = instance_labels[choices]
    semantic_labels = semantic_labels[choices]
    pcl_color = pcl_color[choices]
    instance_points = []
    instance_class = []
    ref_target = []
    ins_obbs = []
    pts_batch = []
    pred_obbs = []
    # print(np.unique(instance_labels))
    nel_count = 0

    for i, gt_id in enumerate(instance_bboxes[:, -1]): # -1 mean id of object in scene, equal to objecid
        if gt_id == object_id:
            gt_bbox = instance_bboxes[i, :]
    
    for i_instance in np.unique(instance_labels):
        # find all points belong to that instance
        ind = np.nonzero(instance_labels == i_instance)[0]
        # find the semantic label
        ins_class = semantic_labels[ind[0]]
        if ins_class in DC:
            
            x = point_cloud[ind]
            ins_class = DC[int(ins_class)]
            instance_class.append(ins_class)

            pc = x[:, :3]
            center = 0.5 * (pc.min(0) + pc.max(0))
            size = pc.max(0) - pc.min(0)
            ins_obb = np.concatenate((center, size, np.array([0])))
            ins_obbs.append(ins_obb)
            x = random_sampling(x, 1024)
            instance_points.append(x)
            if ins_class in nel_label[i]:
                if ins_class != cls[i]:
                    nel_count += 1
            
    num_target.append(np.where(instance_class == cls[i])[0].size)
    num_nel.append(nel_count)
    

{'scene_id': 'scene0011_00', 'object_id': '5', 'object_name': 'chair', 'ann_id': '3', 'description': 'there is a dark brown wooden and leather chair. placed in the table of the kitchen.', 'token': ['there', 'is', 'a', 'dark', 'brown', 'wooden', 'and', 'leather', 'chair', '.', 'placed', 'in', 'the', 'table', 'of', 'the', 'kitchen', '.'], 'id': 0}


In [12]:
num_nel

[4]

In [13]:
ins_obb.shape

(7,)

In [14]:
nel

array(['[2, 4, 17]', '[2, 17]', '[2, 4]', ..., '[8, 17]', '[17]',
       '[8, 17]'], dtype=object)

In [15]:
num_target

[4]

In [16]:
gt_bbox

array([ 0.0316933 , -1.01040626,  0.51449943,  0.5163914 ,  0.53976846,
        0.99540478,  5.        ,  5.        ])

In [17]:
import open3d as o3d
geometries = []
# geometries.append(o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5))
for i in ins_obbs:
    # geometries.append(i[:6].tolist())
    obb = o3d.geometry.OrientedBoundingBox()
    obb.center = i[:3]
    obb.extent = i[3:6]
    obb.color = [1, 0, 0]
    geometries.append(obb)
obb = o3d.geometry.OrientedBoundingBox()
obb.center = gt_bbox[:3]
obb.extent = gt_bbox[3:6]
obb.color = [0, 1, 0]
geometries.append(obb)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(mesh_vertices[:, 0:3])
pcd.colors = o3d.utility.Vector3dVector(mesh_vertices[:, 3:6])

# Visualize
o3d.visualization.draw_geometries([pcd])


o3d.visualization.draw_geometries(geometries)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [34]:
point_cloud.shape

(1024, 6)

In [25]:
def voxelize_box(box: o3d.geometry.OrientedBoundingBox, voxel_size: float):
    # Sample points inside the box by creating a dense 3D grid
    extent = box.extent
    num_x = int(extent[0] / voxel_size)
    num_y = int(extent[1] / voxel_size)
    num_z = int(extent[2] / voxel_size)

    x = np.linspace(-extent[0] / 2, extent[0] / 2, num_x)
    y = np.linspace(-extent[1] / 2, extent[1] / 2, num_y)
    z = np.linspace(-extent[2] / 2, extent[2] / 2, num_z)
    xv, yv, zv = np.meshgrid(x, y, z, indexing='ij')
    grid = np.vstack((xv.ravel(), yv.ravel(), zv.ravel())).T

    # Transform local box points to world coordinates
    points = (box.R @ grid.T).T + box.center
    voxel_coords = np.floor(points / voxel_size).astype(int)

    return set(map(tuple, voxel_coords))

def compute_iou_3d(box1, box2, voxel_size=0.01):
    voxels1 = voxelize_box(box1, voxel_size)
    voxels2 = voxelize_box(box2, voxel_size)

    intersection = voxels1 & voxels2
    union = voxels1 | voxels2

    if not union:
        return 0.0
    return len(intersection) / len(union)

# Create two oriented bounding boxes
center1 = [0.0, 0.0, 0.0]
extent1 = [1.0, 2.0, 1.0]
R = np.eye(3)
box1 = o3d.geometry.OrientedBoundingBox(center=center1, R=R, extent=extent1)

center2 = [0, 0.0, 0.0]
extent2 = [1.0, 2.0, 1.0]
box2 = o3d.geometry.OrientedBoundingBox(center=center2, R=R, extent=extent2)

# Compute IoU
iou = compute_iou_3d(box1, box2, voxel_size=0.02)

In [26]:
iou

1.0